In [1]:
import json, mlflow, mlflow.pytorch, torch, torch.nn as nn
from pathlib import Path
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, Subset

DATA = Path("data/asl_alphabet_train")
print(torch.cuda.is_available(), [p.name for p in DATA.iterdir()][:5])

True ['A', 'B', 'C', 'D', 'del']


In [2]:
P = dict(img_size=128, batch_size=64, lr=1e-3, epochs=8, val_split=0.1,
         subset=None, seed=42)  # subset=None for the full run

dev = "cuda" if torch.cuda.is_available() else "cpu"
tf = transforms.Compose([
    transforms.Resize((P["img_size"], P["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
ds = datasets.ImageFolder(DATA, transform=tf)
classes = ds.classes
print(len(ds), len(classes), classes)

g = torch.Generator().manual_seed(P["seed"])
if P["subset"]:
    ds = Subset(ds, torch.randperm(len(ds), generator=g)[:P["subset"]].tolist())
n_val = int(len(ds) * P["val_split"])
train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val], generator=g)

# num_workers=0 on purpose: multiprocessing workers often hang in Windows notebooks
tl = DataLoader(train_ds, P["batch_size"], shuffle=True, num_workers=0)
vl = DataLoader(val_ds, P["batch_size"], num_workers=0)

87000 29 ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [3]:
# Load pretrained MobileNetV2
m = models.mobilenet_v2(
    weights=models.MobileNet_V2_Weights.DEFAULT
)

# Freeze feature extraction layers
for param in m.features.parameters():
    param.requires_grad = False

# Replace the final classification layer
m.classifier[1] = nn.Linear(
    m.classifier[1].in_features,
    len(classes)
)

m = m.to(dev)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    m.classifier.parameters(),
    lr=P["lr"]
)

In [4]:
def run(loader, training=True):
    if training:
        m.train()
    else:
        m.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images = images.to(dev)
            labels = labels.to(dev)

            if training:
                optimizer.zero_grad()

            outputs = m(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            predicted = outputs.argmax(dim=1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

In [5]:
RUN_NAME = "full-frozen"

mlflow.set_experiment("asl-alphabet")
with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params({**P, "model": "mobilenet_v2", "device": dev, "n_images": len(ds)})
    best = 0.0
    for ep in range(P["epochs"]):
        tr_l, tr_a = run(tl, True)
        va_l, va_a = run(vl, False)
        best = max(best, va_a)
        mlflow.log_metrics({"train_loss": tr_l, "train_acc": tr_a,
                            "val_loss": va_l, "val_acc": va_a}, step=ep)
        print(f"ep{ep} train_acc={tr_a:.3f} val_acc={va_a:.3f}", flush=True)
    mlflow.log_metric("best_val_acc", best)
    json.dump(classes, open("classes.json", "w"))
    mlflow.log_artifact("classes.json")
    mlflow.pytorch.log_model(m, name="model", serialization_format="pickle")

ep0 train_acc=0.810 val_acc=0.904
ep1 train_acc=0.900 val_acc=0.933
ep2 train_acc=0.911 val_acc=0.939
ep3 train_acc=0.919 val_acc=0.943
ep4 train_acc=0.922 val_acc=0.947
ep5 train_acc=0.924 val_acc=0.948
ep6 train_acc=0.927 val_acc=0.950
ep7 train_acc=0.926 val_acc=0.949


2026/11/01 11:33:26 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/11/01 11:33:26 WARNING mlflow.utils.requirements_utils: Found torch version (2.14.1+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.14.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/01 11:33:40 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.29.1+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.29.1' without the loca

In [6]:
mlflow.set_experiment("asl-alphabet")
with mlflow.start_run(run_name="mobilenetv2-frozen-run1"):
    mlflow.log_params({**P, "model": "mobilenet_v2", "device": dev})
    for ep in range(P["epochs"]):
        tr_l, tr_a = run(tl, True)
        va_l, va_a = run(vl, False)
        mlflow.log_metrics({"train_loss": tr_l, "train_acc": tr_a,
                            "val_loss": va_l, "val_acc": va_a}, step=ep)
        print(f"ep{ep} train_acc={tr_a:.3f} val_acc={va_a:.3f}")
    json.dump(classes, open("classes.json", "w"))
    mlflow.log_artifact("classes.json")
    mlflow.pytorch.log_model(m, "model", serialization_format="pickle")

ep0 train_acc=0.929 val_acc=0.950
ep1 train_acc=0.930 val_acc=0.953
ep2 train_acc=0.929 val_acc=0.953
ep3 train_acc=0.932 val_acc=0.956
ep4 train_acc=0.929 val_acc=0.953
ep5 train_acc=0.931 val_acc=0.956
ep6 train_acc=0.931 val_acc=0.955


2026/11/01 12:02:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/01 12:02:48 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/11/01 12:02:48 WARNING mlflow.utils.requirements_utils: Found torch version (2.14.1+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.14.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


ep7 train_acc=0.931 val_acc=0.956


2026/11/01 12:02:58 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.29.1+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.29.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
